In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
import os



In [2]:
df_final = pd.read_pickle('results/df_final.pkl')
df_final.shape

(204942, 50)

In [4]:

# Calculate missing percentage for all features
missing_stats = pd.DataFrame({
    'feature': df_final.columns,
    'missing_count': df_final.isnull().sum().values,
    'missing_pct': (df_final.isnull().sum() / len(df_final) * 100).values
}).sort_values('missing_pct', ascending=False)

# Categorize features by missing percentage
print(f"\nMISSING DATA:")

categories = {
    'Complete (0%)': (0, 0),
    'Low (0-5%)': (0.01, 5),
    'Informative (5-50%)': (5.01, 50),
    'High (50-80%)': (50.01, 80),
    'Very High (80-95%)': (80.01, 95),
    'Almost Complete (95-100%)': (95.01, 100)
}

for cat_name, (lower, upper) in categories.items():
    if lower == 0 and upper == 0:
        mask = missing_stats['missing_pct'] == 0
    else:
        mask = (missing_stats['missing_pct'] >= lower) & (missing_stats['missing_pct'] <= upper)
    
    count = mask.sum()
    print(f"   {cat_name:30s}: {count:>3d} features")


MISSING DATA:
   Complete (0%)                 :  23 features
   Low (0-5%)                    :   9 features
   Informative (5-50%)           :  18 features
   High (50-80%)                 :   0 features
   Very High (80-95%)            :   0 features
   Almost Complete (95-100%)     :   0 features


In [ ]:
# Informative range (5-50% missing) - Create individual indicators
informative_features = missing_stats[
    (missing_stats['missing_pct'] >= 5) & 
    (missing_stats['missing_pct'] <= 50)
]['feature'].tolist()

print(f"\nINFORMATIVE RANGE (5-50% missing): {len(informative_features)} features")

for feat in informative_features:
    missing_pct = missing_stats[missing_stats['feature'] == feat]['missing_pct'].values[0]
    print(f"     {feat:50s} {missing_pct:>6.2f}%")



INFORMATIVE RANGE (5-50% missing): 18 features
     ping_ms                                             28.16%
     datarate                                             7.65%
     jitter                                               7.65%
     Pos in Ref Round                                     6.91%
     PCell_Downlink_bandwidth_MHz                         6.15%
     PCell_Uplink_frequency                               6.15%
     PCell_Cell_Identity                                  6.15%
     PCell_TAC                                            6.15%
     PCell_Cell_ID                                        6.15%
     PCell_Uplink_bandwidth_MHz                           6.15%
     PCell_Downlink_frequency                             6.15%
     PCell_Band_Indicator                                 6.15%
     PCell_Uplink_TB_Size                                 5.96%
     PCell_Uplink_Tx_Power_(dBm)                          5.96%
     PCell_Uplink_Num_RBs                               

In [6]:
for feat in informative_features:
    indicator_name = f'{feat}_missing'
    df_final[indicator_name] = df_final[feat].isnull().astype(int)
    missing_pct = missing_stats[missing_stats['feature'] == feat]['missing_pct'].values[0]
    print(f"  {indicator_name:<60s} ({missing_pct:>5.2f}% missing)")

print(f"\n Created {len(informative_features)} informative indicators")

  ping_ms_missing                                              (28.16% missing)
  datarate_missing                                             ( 7.65% missing)
  jitter_missing                                               ( 7.65% missing)
  Pos in Ref Round_missing                                     ( 6.91% missing)
  PCell_Downlink_bandwidth_MHz_missing                         ( 6.15% missing)
  PCell_Uplink_frequency_missing                               ( 6.15% missing)
  PCell_Cell_Identity_missing                                  ( 6.15% missing)
  PCell_TAC_missing                                            ( 6.15% missing)
  PCell_Cell_ID_missing                                        ( 6.15% missing)
  PCell_Uplink_bandwidth_MHz_missing                           ( 6.15% missing)
  PCell_Downlink_frequency_missing                             ( 6.15% missing)
  PCell_Band_Indicator_missing                                 ( 6.15% missing)
  PCell_Uplink_TB_Size_missing          

In [7]:

# Low missing (0-5%)
low_missing = missing_stats[(missing_stats['missing_pct'] > 0) & (missing_stats['missing_pct'] <= 5)]
print(f"\nLOW MISSING (0-5%): {len(low_missing)} features")
print(f"{'Feature':<50s} {'Missing %':>10s} {'Type'}")

for idx, row in low_missing.iterrows():
    feat = row['feature']
    pct = row['missing_pct']

    print(f"{feat:<50s} {pct:>9.2f}% ")


LOW MISSING (0-5%): 9 features
Feature                                             Missing % Type
PCell_freq_MHz                                          4.89% 
PCell_SNR_1                                             4.89% 
PCell_E-ARFCN                                           4.89% 
PCell_SNR_2                                             4.89% 
PCell_RSSI_max                                          4.89% 
PCell_RSRQ_max                                          4.89% 
PCell_RSRP_max                                          4.89% 
Traffic Jam Factor                                      1.40% 
Altitude                                                0.04% 


In [8]:
signal_quality_features = ['PCell_RSRP_max', 'PCell_RSSI_max','PCell_RSRQ_max', 
                           'PCell_SNR_1', 'PCell_SNR_2','PCell_freq_MHz','PCell_E-ARFCN']
signal_quality_existing = [f for f in signal_quality_features if f in df_final.columns]

print(f"Creating individual indicators for {len(signal_quality_existing)} features...")

for feat in signal_quality_existing:
    indicator_name = f'{feat}_missing'
    df_final[indicator_name] = df_final[feat].isnull().astype(int)
    missing_pct = missing_stats[missing_stats['feature'] == feat]['missing_pct'].values[0]
    print(f"{indicator_name:<60s} ({missing_pct:>5.2f}% missing)")

print(f"\nCreated {len(signal_quality_existing)} signal quality indicators")

Creating individual indicators for 7 features...
PCell_RSRP_max_missing                                       ( 4.89% missing)
PCell_RSSI_max_missing                                       ( 4.89% missing)
PCell_RSRQ_max_missing                                       ( 4.89% missing)
PCell_SNR_1_missing                                          ( 4.89% missing)
PCell_SNR_2_missing                                          ( 4.89% missing)
PCell_freq_MHz_missing                                       ( 4.89% missing)
PCell_E-ARFCN_missing                                        ( 4.89% missing)

Created 7 signal quality indicators


In [9]:
df_engineered = df_final.copy()

In [17]:
all_indicator_cols = [col for col in df_engineered.columns if col.endswith('_missing')]

print(f"\nCREATED INDICATORS:")
print(f"   Informative (5-50%):        {len(informative_features):>3d} indicators")
print(f"   TOTAL:                      {len(all_indicator_cols):>3d} indicators")

# Verify no missing values in indicators
indicator_missing = df_engineered[all_indicator_cols].isnull().sum().sum()

print(f"\nCurrent dataset shape: {df_engineered.shape}")




CREATED INDICATORS:
   Informative (5-50%):         18 indicators
   TOTAL:                       25 indicators

Current dataset shape: (204942, 75)


In [18]:
categorical_cols = df_engineered.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"{'Feature':<30s} {'Unique Values':>15s} {'Type':<15s}")

for col in categorical_cols:
    n_unique = df_engineered[col].nunique()
    dtype = str(df_engineered[col].dtype)
    print(f"{col:<30s} {n_unique:>15d} {dtype:<15s}")

Feature                          Unique Values Type           
device                                       4 object         
Traffic Street Name                         64 object         
area                                         6 object         
PCell_Downlink_bandwidth_MHz                 4 object         
PCell_Uplink_bandwidth_MHz                   4 object         
scenario                                     4 object         
direction                                    2 object         
measured_qos                                 2 object         


In [19]:
# Categorize by cardinality
low_cardinality = []
high_cardinality = []

for col in categorical_cols:
    n_unique = df_engineered[col].nunique()
    if n_unique <= 10:
        low_cardinality.append(col)
    else:
        high_cardinality.append(col)

print(f"\nLOW CARDINALITY (≤10 unique): {len(low_cardinality)} features")
for col in low_cardinality:
    n_unique = df_engineered[col].nunique()
    values = df_engineered[col].value_counts()
    print(f"\n   {col} ({n_unique} unique values):")
    for val, count in values.items():
        print(f"      {val}: {count:,} ({count/len(df_engineered)*100:.2f}%)")

print(f"\nHIGH CARDINALITY : {len(high_cardinality)} features")
for col in high_cardinality:
    n_unique = df_engineered[col].nunique()
    print(f"    {col}: {n_unique} unique values → Consider dropping")


LOW CARDINALITY (≤10 unique): 7 features

   device (4 unique values):
      pc3: 59,723 (29.14%)
      pc1: 59,519 (29.04%)
      pc2: 42,879 (20.92%)
      pc4: 42,821 (20.89%)

   area (6 unique values):
      Park: 73,409 (35.82%)
      Residential: 62,746 (30.62%)
      Avenue: 43,466 (21.21%)
      Highway: 23,482 (11.46%)
      Tunnel: 1,097 (0.54%)
      UNKNOWN: 742 (0.36%)

   PCell_Downlink_bandwidth_MHz (4 unique values):
      20 : 155,911 (76.08%)
      15 : 35,651 (17.40%)
      5 : 712 (0.35%)
      10 : 61 (0.03%)

   PCell_Uplink_bandwidth_MHz (4 unique values):
      20 : 155,911 (76.08%)
      15 : 35,651 (17.40%)
      5 : 712 (0.35%)
      10 : 61 (0.03%)

   scenario (4 unique values):
      A3D: 64,626 (31.53%)
      A3U: 63,355 (30.91%)
      A2U: 42,730 (20.85%)
      A2D: 34,231 (16.70%)

   direction (2 unique values):
      uplink: 106,085 (51.76%)
      downlink: 98,857 (48.24%)

   measured_qos (2 unique values):
      datarate: 127,981 (62.45%)
      de

In [13]:
low_cardinality

['device',
 'area',
 'PCell_Downlink_bandwidth_MHz',
 'PCell_Uplink_bandwidth_MHz',
 'scenario',
 'direction',
 'measured_qos']

In [20]:
#convert to int to float
bandwidth_cols=['PCell_Downlink_bandwidth_MHz',
 'PCell_Uplink_bandwidth_MHz',
 'SCell_Downlink_bandwidth_MHz',
 'SCell_Uplink_bandwidth_MHz']


for col in bandwidth_cols:
    if col in df_engineered.columns:
        df_engineered[col] = pd.to_numeric(df_engineered[col], errors='coerce')


In [21]:
if len(high_cardinality) > 0:
    for col in high_cardinality:
        n_unique = df_engineered[col].nunique()
        print(f" {col} ({n_unique} unique) → Too many categories")
    
    df_engineered = df_engineered.drop(columns=high_cardinality)
    print(f"Dropped {len(high_cardinality)} features")

 Traffic Street Name (64 unique) → Too many categories
Dropped 1 features


In [22]:
low_cad = ['device']

# One-hot encode
df_engineered = pd.get_dummies(
    df_engineered, 
    columns=low_cad, 
    dtype=int
)
 

In [23]:

low_cal=[
 'direction',
 'measured_qos',
]
# One-hot encode low cardinality features

print(f"\n ONE-HOT ENCODING LOW CARDINALITY:")

# Get features to encode (exclude timestamp, date)
features_to_encode = [col for col in low_cal 
                        if col not in ['timestamp', 'ts_gps', 'date']]

print(f"  Encoding {len(features_to_encode)} features with drop_first=True...")

# Store original shape
shape_before = df_engineered.shape

# One-hot encode
df_engineered = pd.get_dummies(df_engineered, 
                                columns=features_to_encode, 
                                drop_first=True,
                                dtype=int)

shape_after = df_engineered.shape



 ONE-HOT ENCODING LOW CARDINALITY:
  Encoding 2 features with drop_first=True...


In [24]:


print(f"  Before: {shape_before[1]} features")
print(f"  After:  {shape_after[1]} features")
print(f"  Added:  {shape_after[1] - shape_before[1]} dummy variables")

# Show created dummy columns
new_cols = [col for col in df_engineered.columns if any(f'{feat}_' in col for feat in features_to_encode)]
print(f"\n  Created dummy variables ({len(new_cols)}):")


  Before: 77 features
  After:  77 features
  Added:  0 dummy variables

  Created dummy variables (2):


In [25]:
new_cols

['direction_uplink', 'measured_qos_delay']

In [26]:
# Binary encode 'operator' (1,2 → 0,1)
if 'operator' in df_engineered.columns:
    df_engineered['operator'] = df_engineered['operator'] - 1



In [27]:


# Check timestamp column
timestamp_col = 'timestamp'
if timestamp_col not in df_engineered.columns:
    print("Error: 'timestamp' column not found")
else:
    print(f" Found timestamp column: {timestamp_col}")
    print(f"  Data type: {df_engineered[timestamp_col].dtype}")
    print(f"  Date range: {df_engineered[timestamp_col].min()} to {df_engineered[timestamp_col].max()}")
    
    # Ensure datetime type
    if not pd.api.types.is_datetime64_any_dtype(df_engineered[timestamp_col]):
        print("\n  Converting to datetime...")
        df_engineered[timestamp_col] = pd.to_datetime(df_engineered[timestamp_col])
        print(f"Converted to: {df_engineered[timestamp_col].dtype}")
    
    # Extract temporal features
    print("\n Extracting temporal features...")
    
    # Hour of day (0-23)
    df_engineered['hour'] = df_engineered[timestamp_col].dt.hour
    print(f"   hour: {df_engineered['hour'].min()}-{df_engineered['hour'].max()} range")
    
    # Day of week (0=Monday, 6=Sunday)
    df_engineered['day_of_week'] = df_engineered[timestamp_col].dt.dayofweek
    print(f"   day_of_week: {df_engineered['day_of_week'].min()}-{df_engineered['day_of_week'].max()} range")



 Found timestamp column: timestamp
  Data type: datetime64[us, Europe/Berlin]
  Date range: 2021-06-22 09:49:54+02:00 to 2021-06-24 18:59:59+02:00

 Extracting temporal features...
   hour: 8-18 range
   day_of_week: 1-3 range


In [28]:


print(f"\n  Hour distribution:")
hour_dist = df_engineered['hour'].value_counts().sort_index()
for hour, count in hour_dist.items():
    print(f"    {hour:02d}:00 - {count:>7,} rows ({count/len(df_engineered)*100:>5.2f}%)")

print(f"\n  Day of week distribution:")
day_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_dist = df_engineered['day_of_week'].value_counts().sort_index()
for dow, count in dow_dist.items():
    print(f"    {dow} ({day_names[dow]:9s}) - {count:>7,} rows ({count/len(df_engineered)*100:>5.2f}%)")


print(f"\n Current dataset shape: {df_engineered.shape}")


  Hour distribution:
    08:00 -   1,096 rows ( 0.53%)
    09:00 -  21,904 rows (10.69%)
    10:00 -  31,036 rows (15.14%)
    11:00 -  29,842 rows (14.56%)
    12:00 -  11,509 rows ( 5.62%)
    13:00 -   2,248 rows ( 1.10%)
    14:00 -  19,454 rows ( 9.49%)
    15:00 -  27,001 rows (13.17%)
    16:00 -  32,927 rows (16.07%)
    17:00 -  19,143 rows ( 9.34%)
    18:00 -   8,782 rows ( 4.29%)

  Day of week distribution:
    1 (Tuesday  ) -  79,564 rows (38.82%)
    2 (Wednesday) -  79,015 rows (38.55%)
    3 (Thursday ) -  46,363 rows (22.62%)

 Current dataset shape: (204942, 79)


In [29]:
# Check if device and operator dummy columns exist
device_cols = [col for col in df_engineered.columns if col.startswith('device_')]
operator_cols = [col for col in df_engineered.columns if col.startswith('operator')]

print(f"   Device dummies:   {len(device_cols)}")
print(f"   Operator dummies: {len(operator_cols)}")

   Device dummies:   4
   Operator dummies: 1


In [30]:
df_engineered.shape

(204942, 79)

In [32]:
df_engineered.info()

<class 'pandas.core.frame.DataFrame'>
Index: 204942 entries, 216 to 207433
Data columns (total 79 columns):
 #   Column                                Non-Null Count   Dtype                        
---  ------                                --------------   -----                        
 0   timestamp                             204942 non-null  datetime64[us, Europe/Berlin]
 1   ping_ms                               147235 non-null  float64                      
 2   datarate                              189263 non-null  float64                      
 3   jitter                                189263 non-null  float64                      
 4   Latitude                              204942 non-null  float64                      
 5   Longitude                             204942 non-null  float64                      
 6   Altitude                              204851 non-null  float64                      
 7   speed_kmh                             204942 non-null  float64               

In [33]:
remove_from_imputation = [
    'PCell_Cell_Identity',    
    'PCell_TAC',                 
    'PCell_E-ARFCN','PCell_Uplink_frequency','PCell_Cell_ID'
]

In [34]:
df_engineered = df_engineered.drop(columns=remove_from_imputation)

In [35]:


log_features = [
    'datarate',
    'target_datarate',
    'PCell_Downlink_TB_Size',
    'PCell_Uplink_TB_Size',
    'PCell_DL_RBs_MCS_Low', 
    'PCell_DL_RBs_MCS_Mid', 'PCell_DL_RBs_MCS_High',
    'PCell_Downlink_Num_RBs', 'PCell_Uplink_Num_RBs'
    ,'ping_ms','Pos in Ref Round','Traffic Distance','speed_kmh','PCell_Uplink_Tx_Power_(dBm)'
]

print(f"\nApplying log(x+1) to {len(log_features)} features...")

for feat in log_features:
    if feat in df_engineered.columns:
        # Before
        before = df_engineered[feat].dropna()
        print(f"\n{feat}:")
        print(f"  Before: min={before.min():.0f}, max={before.max():.0f}")
        
        # Apply log(x + 1)
        df_engineered[feat] = np.log1p(df_engineered[feat])
        
        # After
        after = df_engineered[feat].dropna()
        print(f"  After:  min={after.min():.2f}, max={after.max():.2f}")





Applying log(x+1) to 14 features...

datarate:
  Before: min=851, max=271000000
  After:  min=6.75, max=19.42

target_datarate:
  Before: min=400000, max=350000000
  After:  min=12.90, max=19.67

PCell_Downlink_TB_Size:
  Before: min=7, max=18679098
  After:  min=2.08, max=16.74

PCell_Uplink_TB_Size:
  Before: min=0, max=6211206
  After:  min=0.00, max=15.64

PCell_DL_RBs_MCS_Low:
  Before: min=0, max=92588
  After:  min=0.00, max=11.44

PCell_DL_RBs_MCS_Mid:
  Before: min=0, max=95920
  After:  min=0.00, max=11.47

PCell_DL_RBs_MCS_High:
  Before: min=0, max=100376
  After:  min=0.00, max=11.52

PCell_Downlink_Num_RBs:
  Before: min=1, max=100580
  After:  min=0.69, max=11.52

PCell_Uplink_Num_RBs:
  Before: min=1, max=94359
  After:  min=0.69, max=11.45

ping_ms:
  Before: min=16, max=110676
  After:  min=2.83, max=11.61

Pos in Ref Round:
  Before: min=0, max=17319
  After:  min=0.00, max=9.76

Traffic Distance:
  Before: min=0, max=268
  After:  min=0.00, max=5.60

speed_kmh:
  B

In [36]:
df_engineered[log_features]

,datarate,target_datarate,PCell_Downlink_TB_Size,PCell_Uplink_TB_Size,PCell_DL_RBs_MCS_Low,PCell_DL_RBs_MCS_Mid,PCell_DL_RBs_MCS_High,PCell_Downlink_Num_RBs,PCell_Uplink_Num_RBs,ping_ms,Pos in Ref Round,Traffic Distance,speed_kmh,PCell_Uplink_Tx_Power_(dBm)
216,18.045260,19.673444,15.293008,7.726654,3.367296,0.000000,10.305179,10.306115,6.063785,NaN,NaN,3.953266,0.000000,4.720503
217,17.014184,19.673444,14.667208,9.902787,4.653960,7.403670,10.591497,10.634412,6.287859,NaN,NaN,3.920668,0.000000,1.540715
218,17.723526,19.673444,14.897844,10.036925,2.564949,8.722091,10.536274,10.687503,6.204558,NaN,NaN,3.585666,0.000000,2.404548
219,18.071123,19.673444,15.028756,7.693026,2.833213,4.262680,10.322329,10.325154,6.040255,7.242082,NaN,3.716122,0.000000,4.809736
220,16.850464,19.673444,14.227611,9.690047,4.653960,7.603399,10.172827,10.250264,6.068426,NaN,NaN,3.918528,0.000000,1.664227
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
207429,18.997294,19.673444,15.964078,7.651596,4.158883,0.000000,10.986343,10.987409,5.476464,NaN,NaN,3.687761,1.329195,4.726409
207430,18.997294,19.673444,15.785832,7.871311,2.079442,0.000000,10.747294,10.747444,5.509388,NaN,NaN,3.691173,0.554345,4.713535
207431,18.921456,19.673444,15.736908,7.824446,4.454347,0.000000,10.763102,10.764900,5.505332,NaN,NaN,3.694615,0.000000,4.719067
207432,18.991660,19.673444,16.254759,7.827241,5.225747,0.000000,11.237778,11.240211,5.556828,6.293419,NaN,3.703097,0.000000,4.718305


In [ ]:
# Save as pickle
df_engineered.to_pickle('results/df_engineered.pkl')
print("\n Saved: results/df_engineered.pkl")

# Save as CSV 
df_engineered.to_csv('results/df_engineered.csv', index=False)
print(" Saved: results/df_engineered.csv")

df_engineered.to_parquet('results/df_engineered.parquet')


 Saved: results/df_engineered.pkl
 Saved: results/df_engineered.csv
